# PROYECTO PTIA 2026-2
## StudyFlow AI: capa de asignación

**Integrantes:**

Karol Ximena Rodriguez Reyes
Juan David Moreno D'Aleman

---

### ¿Qué vamos a hacer en este notebook?

`LLM (entiende lo que pide el estudiante)` -> `Validador (¿cabe en el tiempo?)` -> `MLP (prioriza y reparte)` -> **`Capa de asignación (arma las sesiones)`** -> `Validador de salida + LLM redactor`

El MLP solo entrega **dos resultados por actividad**: una prioridad (0 a 1) y un perfil de reparto en 3 tramos. No sabe de días, horas ni franjas, así que todavía no hay horario. Aquí desarrollamos la capa **determinista** que convierte esos resultados en sesiones concretas `(actividad, día, hora inicio, hora fin)` y garantiza que ninguna sesión viole la disponibilidad ni la fecha límite.

Se prueba aquí antes de pasarla a archivos `.py`. Flujo: `plan validado + salidas del MLP -> asignar -> sesiones + resumen por actividad -> front y LLM redactor`.

Todo el código, a excepción de las pruebas, se mudará a archivos `.py` para conectar los módulos entre sí y el front con el back.

# PASO 0. DEPENDENCIAS

Las tres celdas siguientes son copia **exactas** de `models.py`, `disponibilidad.py` y `validador.py`, tal como estan en `StudyFlow_CapaValidacion.ipynb`. Están aquí solo para que este notebook corra solo; al migrar se reemplazan por `import`.

In [1]:
"""Modelos de entrada (forma y valores). Ver StudyFlow_CapaValidacion.ipynb, paso 2."""
from datetime import date, time
from enum import Enum, IntEnum

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


class Nivel(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class DuracionSesion(IntEnum):
    M30 = 30
    M60 = 60
    M90 = 90
    M120 = 120
    M150 = 150
    M180 = 180
    M210 = 210
    M240 = 240


class Actividad(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    nombre: str = Field(min_length=1, max_length=100)
    prioridad: Nivel
    dificultad: Nivel
    fecha_limite: date
    horas_estimadas: float = Field(gt=0, le=100)

    @field_validator("prioridad", "dificultad", mode="before")
    @classmethod
    def _nivel_sin_mayusculas(cls, v):
        return v.strip().lower() if isinstance(v, str) else v


class Franja(BaseModel):
    inicio: time
    fin: time

    @model_validator(mode="after")
    def _fin_despues_de_inicio(self):
        if self.fin <= self.inicio:
            raise ValueError("la franja debe terminar después de empezar")
        return self


class DisponibilidadDia(BaseModel):
    dia: date
    franjas: list[Franja]

    @model_validator(mode="after")
    def _sin_traslapes(self):
        ordenadas = sorted(self.franjas, key=lambda f: f.inicio)
        for a, b in zip(ordenadas, ordenadas[1:]):
            if b.inicio < a.fin:
                raise ValueError(f"las franjas {a.inicio}-{a.fin} y {b.inicio}-{b.fin} se traslapan")
        return self


class PlanEntrada(BaseModel):
    actividades: list[Actividad] = Field(min_length=1)
    disponibilidad: list[DisponibilidadDia] = Field(min_length=1)
    max_horas_dia: float | None = Field(default=None, gt=0, le=8)
    duracion_sesion: DuracionSesion

    @model_validator(mode="after")
    def _sin_dias_repetidos(self):
        dias = [d.dia for d in self.disponibilidad]
        if len(dias) != len(set(dias)):
            raise ValueError("hay días repetidos en la disponibilidad")
        return self

In [2]:
"""Capacidad efectiva de estudio por día. Ver StudyFlow_CapaValidacion.ipynb, paso 3."""
from datetime import date, time

BLOQUE_MIN = 30  # unidad de capacidad y duración mínima de una sesión (minutos)


def _minutos(t: time):
    return t.hour * 60 + t.minute


def capacidad_dia(dia: DisponibilidadDia, max_horas_dia: float | None = None):
    minutos = sum((_minutos(f.fin) - _minutos(f.inicio)) // BLOQUE_MIN * BLOQUE_MIN for f in dia.franjas)
    if max_horas_dia is not None:
        minutos = min(minutos, int(max_horas_dia * 60) // BLOQUE_MIN * BLOQUE_MIN)
    return minutos / 60


def capacidad_entre(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    """Horas efectivas de los días d con inicio <= d < fin (el día `fin` no cuenta)."""
    return sum(capacidad_dia(d, max_horas_dia) for d in disponibilidad if inicio <= d.dia < fin)


def dias_con_capacidad(disponibilidad: list[DisponibilidadDia], max_horas_dia: float | None, inicio: date, fin: date):
    return sum(1 for d in disponibilidad if inicio <= d.dia < fin and capacidad_dia(d, max_horas_dia) > 0)

In [3]:
"""Validación de coherencia del plan (R1-R7). Ver StudyFlow_CapaValidacion.ipynb, paso 4."""
from collections import Counter
from datetime import date
from enum import Enum
from typing import Any, Literal

from pydantic import BaseModel, Field

MARGEN_HORAS = 0.0
VENTANA_MINIMA_DIAS = 3
_ORDEN_PRIORIDAD = {Nivel.BAJA: 0, Nivel.MEDIA: 1, Nivel.ALTA: 2}


class Estado(str, Enum):
    FACTIBLE = "factible"
    CON_ADVERTENCIAS = "factible_con_advertencias"
    INVIABLE = "inviable"


class Hallazgo(BaseModel):
    codigo: str
    mensaje: str
    datos: dict[str, Any] = Field(default_factory=dict)


class Resultado(BaseModel):
    estado: Estado
    errores: list[Hallazgo]
    advertencias: list[Hallazgo]


def _sugerir_recorte(vencen, deficit):
    candidatas = sorted(vencen, key=lambda a: (_ORDEN_PRIORIDAD[a.prioridad], -a.horas_estimadas))
    elegidas, recortado = [], 0.0
    for a in candidatas:
        if recortado >= deficit:
            break
        elegidas.append({"nombre": a.nombre, "horas": a.horas_estimadas})
        recortado += a.horas_estimadas
    return elegidas


def validar(plan: PlanEntrada, fecha_inicio: date, modo: Literal["nuevo", "replanificacion"] = "nuevo") -> Resultado:
    errores: list[Hallazgo] = []
    advertencias: list[Hallazgo] = []
    disp, tope = plan.disponibilidad, plan.max_horas_dia
    vigentes = [a for a in plan.actividades if a.fecha_limite >= fecha_inicio]

    if tope is not None and plan.duracion_sesion / 60 > tope:  # R3a
        errores.append(Hallazgo(
            codigo="R3A_SESION_MAYOR_MAXIMO",
            mensaje=f"La sesión máxima ({int(plan.duracion_sesion)} min) supera el máximo diario ({tope:g} h)",
            datos={"duracion_sesion_min": int(plan.duracion_sesion), "max_horas_dia": tope}))

    for a in plan.actividades:  # R4
        if a.fecha_limite < fecha_inicio:
            errores.append(Hallazgo(
                codigo="R4_FECHA_PASADA",
                mensaje=f"'{a.nombre}' vence el {a.fecha_limite}, antes del inicio del plan ({fecha_inicio})",
                datos={"actividad": a.nombre, "fecha_limite": a.fecha_limite.isoformat()}))

    pasados = sorted(d.dia for d in disp if d.dia < fecha_inicio)  # R5
    if pasados:
        advertencias.append(Hallazgo(
            codigo="R5_DISPONIBILIDAD_PASADA",
            mensaje=f"Se ignoran {len(pasados)} día(s) de disponibilidad anteriores al inicio del plan",
            datos={"dias": [d.isoformat() for d in pasados]}))

    repetidos = sorted(n for n, c in Counter(a.nombre.casefold() for a in plan.actividades).items() if c > 1)  # R6
    if repetidos:
        advertencias.append(Hallazgo(
            codigo="R6_NOMBRE_DUPLICADO",
            mensaje=f"Hay actividades con el mismo nombre: {', '.join(repetidos)}",
            datos={"nombres": repetidos}))

    for d in sorted(disp, key=lambda d: d.dia):  # R3b
        if d.dia >= fecha_inicio and d.franjas and capacidad_dia(d) == 0:
            advertencias.append(Hallazgo(
                codigo="R3B_DIA_SIN_CAPACIDAD",
                mensaje=f"El {d.dia} no aporta horas: todas sus franjas duran menos de 30 min",
                datos={"dia": d.dia.isoformat()}))

    for a in vigentes:  # R2
        n = dias_con_capacidad(disp, tope, fecha_inicio, a.fecha_limite)
        if n < VENTANA_MINIMA_DIAS:
            hallazgo = Hallazgo(
                codigo="R2_VENTANA_MINIMA",
                mensaje=f"'{a.nombre}' tiene {n} día(s) con disponibilidad antes de la entrega; el mínimo es {VENTANA_MINIMA_DIAS}",
                datos={"actividad": a.nombre, "dias_con_disponibilidad": n, "minimo": VENTANA_MINIMA_DIAS})
            (errores if modo == "nuevo" else advertencias).append(hallazgo)

    for limite in sorted({a.fecha_limite for a in vigentes}):  # R1 (cubre R7)
        vencen = [a for a in vigentes if a.fecha_limite <= limite]
        carga = sum(a.horas_estimadas for a in vencen)
        capacidad = capacidad_entre(disp, tope, fecha_inicio, limite)
        deficit = carga - (capacidad + MARGEN_HORAS)
        if deficit > 1e-9:
            errores.append(Hallazgo(
                codigo="R1_CAPACIDAD_INSUFICIENTE",
                mensaje=f"Hasta el {limite} hay {carga:g} h de trabajo y solo {capacidad:g} h disponibles (faltan {deficit:g} h)",
                datos={"fecha_critica": limite.isoformat(), "horas_requeridas": carga, "horas_disponibles": capacidad,
                       "deficit_horas": round(deficit, 4), "sugerencia_recorte": _sugerir_recorte(vencen, deficit)}))

    estado = Estado.INVIABLE if errores else Estado.CON_ADVERTENCIAS if advertencias else Estado.FACTIBLE
    return Resultado(estado=estado, errores=errores, advertencias=advertencias)

# PASO 1. REGLAS Y DECISIONES

## 1.1 Entradas y salidas

| | Qué es |
|---|---|
| Entrada | `PlanEntrada` ya validado, `fecha_inicio`, una `Prediccion` por actividad y, opcional, `ocupadas` (sesiones ya cumplidas o fijadas) |
| Salida | `ResultadoAsignacion`: la lista de `Sesion` y un resumen por actividad (`asignada`, `parcial` o `no_asignada`, con horas asignadas y faltantes) |

Las predicciones se alinean con las actividades **por posición**, no por nombre: el validador solo advierte (R6) cuando hay nombres repetidos.

## 1.2 Cómo trabaja el algoritmo

Se procesa una actividad a la vez, de mayor a menor prioridad, y cada una marca como ocupados los bloques que toma.

1. **Orden:** prioridad del MLP de mayor a menor; a igualdad, fecha límite más cercana y luego nombre (el resultado es reproducible). La más prioritaria escoge primero.
2. **Bloques totales:** `horas_estimadas` entre 30 min, redondeado al bloque más cercano (mínimo 1).
3. **Ventana:** los días desde `fecha_inicio` hasta antes de la fecha límite que **todavía tengan bloques libres** (después de lo que tomaron las actividades anteriores). Se parten en 3 tramos consecutivos y el sobrante va a los primeros: 7 días dan 3, 2 y 2.
4. **Bloques por tramo:** `perfil × bloques totales`, redondeado con el **método del mayor residuo**: cada tramo recibe su parte entera y los bloques que faltan se dan a los tramos con mayor parte decimal. La suma queda exacta y el ajuste se lo lleva el tramo que mejor lo absorbe.

   Ejemplo: 8 h = 16 bloques, perfil `[0.45, 0.35, 0.20]`. Lo ideal es 7.2, 5.6 y 3.2 bloques; con partes enteras 7, 5 y 3 sobra 1 bloque, que va al tramo de 0.6. Resultado: 7, 6 y 3 bloques = **3 h 30, 3 h y 1 h 30**.
5. **Capacidad del tramo:** si un tramo no tiene bloques libres suficientes, se queda con lo que cabe y el excedente pasa al **tramo vecino más cercano con espacio**; a igual distancia, al **posterior**. Un tramo sin días (ventana corta) entrega todo a sus vecinos.
6. **Reparto en días:** dentro de un tramo, bloque a bloque al día con menos carga de esta actividad que aún tenga espacio (empate: el más temprano). Es un reparto parejo que respeta lo que ya ocuparon las actividades de mayor prioridad.
7. **Sesiones:** en cada día se toman los bloques libres más tempranos. Los bloques consecutivos se unen en sesiones de máximo `duracion_sesion` y la última queda más corta (mínimo 30 min). Una sesión nunca cruza el hueco entre dos franjas.
8. **Lo que no cabe** se reporta, nunca se fuerza: `parcial` (se asigna lo que cabe y se informan las horas faltantes) o `no_asignada`.

## 1.3 Bloques libres de un día

Es el mismo cálculo de capacidad del validador: cada franja se redondea hacia abajo a bloques de 30 min contados desde su inicio y se respeta `max_horas_dia`. Para replanificar, se descuentan los bloques que toca una sesión de `ocupadas`, y esas horas **también cuentan para el máximo diario**.

## 1.4 Decisiones

* "Hacia adelante" en el excedente significa **hacia tramos posteriores** del calendario; nunca se pasa de la fecha límite porque la ventana termina antes de ella.
* Horas que no son múltiplo de 30 min (por ejemplo 2.2 h) se redondean al bloque más cercano; `horas_solicitadas` conserva el valor original y `horas_asignadas` lo que se ubicó.
* La capa **no vuelve a validar** el plan (asume que pasó por el validador), pero nunca ubica una sesión fuera de las franjas, antes de `fecha_inicio` ni desde el día de la fecha límite. El paso 5 lo comprueba en cada caso.
* La ventana se calcula con los días que tienen espacio **en ese momento**: una actividad de menor prioridad reparte su trabajo entre lo que dejaron las anteriores.

## PASO 2. MODELOS DE LA CAPA (`asignacion.py`, primera parte)

| Modelo | Reglas / uso |
|---|---|
| `Prediccion` | salida del MLP para una actividad: `prioridad` entre 0 y 1; `perfil` de 3 proporciones no negativas que suman 1 (tolerancia 0.001, por el redondeo del softmax) |
| `Sesion` | `actividad`, `dia`, `inicio`, `fin`. Es la salida y también la entrada `ocupadas` |
| `EstadoAsignacion` | `asignada`, `parcial`, `no_asignada` |
| `ResumenActividad` | nombre, prioridad, horas solicitadas, asignadas y faltantes, estado |
| `ResultadoAsignacion` | sesiones (ordenadas por día y hora) y resumen por actividad (en orden de prioridad) |

In [4]:
"""Modelos de la capa de asignación. Ver StudyFlow_CapaAsignacion.ipynb, paso 2."""
from datetime import date, time
from enum import Enum

from pydantic import BaseModel, Field, field_validator

TRAMOS = 3


class Prediccion(BaseModel):
    prioridad: float = Field(ge=0, le=1)
    perfil: list[float] = Field(min_length=TRAMOS, max_length=TRAMOS)

    @field_validator("perfil")
    @classmethod
    def _perfil_valido(cls, v):
        if min(v) < 0 or abs(sum(v) - 1) > 1e-3:
            raise ValueError("el perfil debe tener proporciones >= 0 que sumen 1")
        return v


class Sesion(BaseModel):
    actividad: str
    dia: date
    inicio: time
    fin: time


class EstadoAsignacion(str, Enum):
    ASIGNADA = "asignada"
    PARCIAL = "parcial"
    NO_ASIGNADA = "no_asignada"


class ResumenActividad(BaseModel):
    nombre: str
    prioridad: float
    horas_solicitadas: float
    horas_asignadas: float
    horas_faltantes: float
    estado: EstadoAsignacion


class ResultadoAsignacion(BaseModel):
    sesiones: list[Sesion]
    actividades: list[ResumenActividad]

## 2.1 Pruebas de los modelos

In [5]:
from pydantic import ValidationError


def rechaza(fn, fragmento):
    try:
        fn()
    except ValidationError as e:
        assert fragmento in str(e), str(e)
        return
    raise AssertionError(f"debía rechazar ({fragmento})")


franjas = lambda *pares: [{"inicio": i, "fin": f} for i, f in pares]


def dia(fecha, *pares):
    return DisponibilidadDia(dia=fecha, franjas=franjas(*pares))


p = Prediccion(prioridad=0.87, perfil=[0.45, 0.35, 0.20])
assert p.prioridad == 0.87
Prediccion(prioridad=0, perfil=[1, 0, 0])
Prediccion(prioridad=1, perfil=[0.333333, 0.333333, 0.333334])
Prediccion(prioridad=0.5, perfil=[0.2999999, 0.7, 0.0000001])      # softmax de Keras: suma 1 con error ínfimo
rechaza(lambda: Prediccion(prioridad=1.2, perfil=[0.5, 0.3, 0.2]), "less_than_equal")
rechaza(lambda: Prediccion(prioridad=-0.1, perfil=[0.5, 0.3, 0.2]), "greater_than_equal")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.5, 0.5]), "too_short")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.4, 0.3, 0.2, 0.1]), "too_long")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[0.5, 0.5, 0.5]), "sumen 1")
rechaza(lambda: Prediccion(prioridad=0.5, perfil=[1.2, -0.2, 0.0]), "sumen 1")

s = Sesion(actividad="Taller", dia="2026-10-26", inicio="18:00", fin="19:00")
assert s.dia == date(2026, 10, 26) and s.fin.hour == 19

print("Modelos de la asignación: todas las pruebas pasaron")

Modelos de la asignación: todas las pruebas pasaron


# PASO 3. ALGORITMO DE ASIGNACIÓN (`asignacion.py`, segunda parte)

Cada paso del apartado 1.2 es una función pequeña, para probarlas por separado y para que `asignar` se lea de arriba abajo:

| Función | Paso | Qué hace |
|---|---|---|
| `_bloques_dia` | 1.3 | bloques libres de un día (franja más temprana primero) |
| `_ordenar` | 1 | orden de las actividades |
| `_repartir_tramos` | 4 | bloques por tramo (mayor residuo) |
| `_dividir_dias` | 3 | parte la ventana en 3 tramos |
| `_ajustar_a_capacidad` | 5 | recorta a lo que cabe y traslada el excedente |
| `_repartir_en_dias` | 6 | reparto parejo entre los días de un tramo |
| `_a_sesiones` | 7 | une bloques consecutivos en sesiones |
| `asignar` | todo | función que usan los demás módulos |

In [6]:
"""Capa de asignación determinista. Ver StudyFlow_CapaAsignacion.ipynb, paso 3."""
from datetime import date, time
from math import floor


def _hora(minutos: int):
    return time(minutos // 60, minutos % 60)


def _bloques_dia(dia: DisponibilidadDia, tope: float | None, ocupadas: list[Sesion]):
    """Inicio (minutos del día) de los bloques libres, del más temprano al más tardío."""
    bloques = []
    for f in sorted(dia.franjas, key=lambda f: f.inicio):
        ini, fin = _minutos(f.inicio), _minutos(f.fin)
        bloques += range(ini, ini + (fin - ini) // BLOQUE_MIN * BLOQUE_MIN, BLOQUE_MIN)
    libres = [b for b in bloques
              if not any(_minutos(s.inicio) < b + BLOQUE_MIN and b < _minutos(s.fin) for s in ocupadas)]
    if tope is not None:
        usados = sum((_minutos(s.fin) - _minutos(s.inicio)) // BLOQUE_MIN for s in ocupadas)
        libres = libres[:max(int(tope * 60) // BLOQUE_MIN - usados, 0)]
    return libres


def _ordenar(actividades: list[Actividad], predicciones: list[Prediccion]):
    """Índices de las actividades de mayor a menor prioridad (empate: fecha límite y nombre)."""
    return sorted(range(len(actividades)),
                  key=lambda i: (-predicciones[i].prioridad, actividades[i].fecha_limite, actividades[i].nombre))


def _repartir_tramos(total: int, perfil: list[float]) -> list[int]:
    """Bloques por tramo con el método del mayor residuo; la suma es exactamente `total`."""
    suma = sum(perfil)
    ideal = [p / suma * total for p in perfil]
    base = [floor(x + 1e-9) for x in ideal]
    por_dar = sorted(range(TRAMOS), key=lambda i: (-(ideal[i] - base[i]), i))
    for i in por_dar[:total - sum(base)]:
        base[i] += 1
    return base


def _dividir_dias(dias: list[date]):
    """Parte los días en 3 tramos consecutivos; los primeros reciben el sobrante."""
    n, resto = divmod(len(dias), TRAMOS)
    tramos, i = [], 0
    for t in range(TRAMOS):
        k = n + (t < resto)
        tramos.append(dias[i:i + k])
        i += k
    return tramos


def _ajustar_a_capacidad(quieren: list[int], caben: list[int]):
    """Recorta cada tramo a su capacidad y pasa el excedente al vecino más cercano con espacio
    (a igual distancia, el posterior). Devuelve (bloques por tramo, bloques sin ubicar)."""
    puestos = [min(q, c) for q, c in zip(quieren, caben)]
    sobras = [q - p for q, p in zip(quieren, puestos)]
    sin_ubicar = 0
    for i, sobra in enumerate(sobras):
        for j in sorted(range(TRAMOS), key=lambda j: (abs(j - i), -j)):
            dar = min(sobra, caben[j] - puestos[j])
            puestos[j] += dar
            sobra -= dar
        sin_ubicar += sobra
    return puestos, sin_ubicar


def _repartir_en_dias(cantidad: int, libres: dict[date, int]):
    """Reparte `cantidad` bloques parejo entre los días, sin pasar el espacio libre de cada uno."""
    puestos = dict.fromkeys(libres, 0)
    for _ in range(cantidad):
        dia = min((d for d in libres if puestos[d] < libres[d]), key=lambda d: (puestos[d], d))
        puestos[dia] += 1
    return puestos


def _a_sesiones(nombre: str, dia: date, bloques: list[int], max_bloques: int):
    """Une bloques consecutivos en sesiones de a lo sumo `max_bloques` bloques."""
    sesiones, inicio, n = [], 0, 0
    for b in bloques + [None]:
        if n and (b is None or b != inicio + n * BLOQUE_MIN or n == max_bloques):
            sesiones.append(Sesion(actividad=nombre, dia=dia, inicio=_hora(inicio), fin=_hora(inicio + n * BLOQUE_MIN)))
            n = 0
        if b is not None:
            if n == 0:
                inicio = b
            n += 1
    return sesiones


def asignar(plan: PlanEntrada, predicciones: list[Prediccion], fecha_inicio: date,
            ocupadas: list[Sesion] | None = None) -> ResultadoAsignacion:
    if len(predicciones) != len(plan.actividades):
        raise ValueError("debe haber una predicción por actividad, en el mismo orden")
    ocupadas = ocupadas or []
    libres = {d.dia: _bloques_dia(d, plan.max_horas_dia, [s for s in ocupadas if s.dia == d.dia])
              for d in plan.disponibilidad if d.dia >= fecha_inicio}
    max_bloques = plan.duracion_sesion // BLOQUE_MIN
    sesiones, resumen = [], []

    for i in _ordenar(plan.actividades, predicciones):
        a, p = plan.actividades[i], predicciones[i]
        total = max(1, int(a.horas_estimadas * 60 / BLOQUE_MIN + 0.5))
        tramos = _dividir_dias(sorted(d for d, b in libres.items() if b and d < a.fecha_limite))
        caben = [sum(len(libres[d]) for d in t) for t in tramos]
        puestos, sin_ubicar = _ajustar_a_capacidad(_repartir_tramos(total, p.perfil), caben)

        for tramo, n in zip(tramos, puestos):
            for dia, k in _repartir_en_dias(n, {d: len(libres[d]) for d in tramo}).items():
                tomados, libres[dia] = libres[dia][:k], libres[dia][k:]
                sesiones += _a_sesiones(a.nombre, dia, tomados, max_bloques)

        asignados = total - sin_ubicar
        estado = (EstadoAsignacion.ASIGNADA if sin_ubicar == 0 else
                  EstadoAsignacion.NO_ASIGNADA if asignados == 0 else EstadoAsignacion.PARCIAL)
        resumen.append(ResumenActividad(
            nombre=a.nombre, prioridad=p.prioridad, horas_solicitadas=a.horas_estimadas,
            horas_asignadas=asignados * BLOQUE_MIN / 60, horas_faltantes=sin_ubicar * BLOQUE_MIN / 60, estado=estado))

    sesiones.sort(key=lambda s: (s.dia, s.inicio))
    return ResultadoAsignacion(sesiones=sesiones, actividades=resumen)

## 3.1 Pruebas de las funciones auxiliares

Los valores esperados están calculados a mano. El caso de `_repartir_tramos(16, [0.45, 0.35, 0.20])` es el ejemplo del apartado 1.2.

In [7]:
import random
from datetime import date

H = lambda h, m=0: h * 60 + m


# _bloques_dia: franjas, redondeo, tope diario y ocupadas
def bl(*pares, tope=None, ocupadas=()):
    return _bloques_dia(dia("2026-10-26", *pares), tope, list(ocupadas))


assert bl(("18:00", "19:30")) == [H(18), H(18, 30), H(19)]
assert bl(("18:00", "18:50")) == [H(18)]                                       # 50 min -> 1 bloque
assert bl(("19:00", "20:00"), ("16:00", "17:00")) == [H(16), H(16, 30), H(19), H(19, 30)]   # franja más temprana primero
assert bl(("18:00", "21:00"), tope=1) == [H(18), H(18, 30)]
assert len(bl(("18:00", "21:00"), tope=2.75)) == 5                             # el tope también va a bloques
ocup = [Sesion(actividad="x", dia="2026-10-26", inicio="18:00", fin="19:00")]
assert bl(("18:00", "21:00"), ocupadas=ocup) == [H(19), H(19, 30), H(20), H(20, 30)]
assert bl(("18:00", "21:00"), tope=2, ocupadas=ocup) == [H(19), H(19, 30)]     # lo ocupado cuenta para el tope
assert bl(("18:00", "21:00"), tope=1, ocupadas=ocup) == []
for pares, tope in [((("11:00", "14:00"), ("16:00", "17:30")), None), ((("11:00", "14:00"), ("16:00", "17:30")), 4),
                    ((("18:00", "18:45"), ("19:00", "19:45")), None), ((("18:00", "18:20"),), None)]:
    assert len(bl(*pares, tope=tope)) / 2 == capacidad_dia(dia("2026-10-26", *pares), tope)   # igual que el validador

# _ordenar: prioridad, luego fecha límite, luego nombre
acts = [Actividad(nombre=n, prioridad="media", dificultad="media", fecha_limite=fl, horas_estimadas=1)
        for n, fl in [("C", "2026-11-03"), ("A", "2026-11-05"), ("B", "2026-11-03")]]
preds = [Prediccion(prioridad=q, perfil=[1/3] * 3) for q in (0.5, 0.5, 0.9)]
assert _ordenar(acts, preds) == [2, 0, 1]
assert _ordenar(acts, [Prediccion(prioridad=0.5, perfil=[1/3] * 3)] * 3) == [2, 0, 1]   # todo empatado: fecha y nombre (B < C)

# _repartir_tramos: mayor residuo
assert _repartir_tramos(16, [0.45, 0.35, 0.20]) == [7, 6, 3]       # 3 h 30, 3 h, 1 h 30
assert _repartir_tramos(8, [0.5, 0.3, 0.2]) == [4, 2, 2]
assert _repartir_tramos(8, [1, 0, 0]) == [8, 0, 0]
assert _repartir_tramos(1, [0.2, 0.5, 0.3]) == [0, 1, 0]
assert _repartir_tramos(8, [1/3] * 3) == [3, 3, 2]                  # residuos iguales: el tramo más temprano primero
random.seed(0)
for _ in range(5000):
    total = random.randint(1, 200)
    w = [random.random() for _ in range(3)]
    perfil = [x / sum(w) for x in w]
    r = _repartir_tramos(total, perfil)
    assert sum(r) == total and min(r) >= 0 and all(abs(x - q * total) < 1 for x, q in zip(r, perfil))

# _dividir_dias
ds = [date(2026, 10, 25 + i) for i in range(7)]
assert [len(t) for t in _dividir_dias(ds)] == [3, 2, 2]
assert [len(t) for t in _dividir_dias(ds[:6])] == [2, 2, 2]
assert [len(t) for t in _dividir_dias(ds[:3])] == [1, 1, 1]
assert [len(t) for t in _dividir_dias(ds[:2])] == [1, 1, 0]
assert [len(t) for t in _dividir_dias(ds[:1])] == [1, 0, 0]
assert [len(t) for t in _dividir_dias([])] == [0, 0, 0]
assert _dividir_dias(ds)[1] == ds[3:5]                              # tramos consecutivos

# _ajustar_a_capacidad
assert _ajustar_a_capacidad([7, 6, 3], [10, 10, 10]) == ([7, 6, 3], 0)
assert _ajustar_a_capacidad([4, 2, 2], [2, 12, 12]) == ([2, 4, 2], 0)     # el excedente del primero va al segundo
assert _ajustar_a_capacidad([2, 2, 4], [12, 12, 2]) == ([2, 4, 2], 0)     # el del último va al anterior
assert _ajustar_a_capacidad([3, 3, 2], [12, 2, 12]) == ([3, 2, 3], 0)     # a igual distancia, al posterior
assert _ajustar_a_capacidad([5, 0, 0], [2, 0, 3]) == ([2, 0, 3], 0)       # salta un tramo sin espacio
assert _ajustar_a_capacidad([4, 2, 2], [3, 0, 0]) == ([3, 0, 0], 5)       # lo que no cabe en ninguno queda sin ubicar
assert _ajustar_a_capacidad([8, 0, 0], [0, 0, 0]) == ([0, 0, 0], 8)

# _repartir_en_dias
d1, d2, d3 = ds[:3]
assert _repartir_en_dias(7, {d1: 6, d2: 6}) == {d1: 4, d2: 3}
assert _repartir_en_dias(3, {d1: 6, d2: 1, d3: 6}) == {d1: 1, d2: 1, d3: 1}
assert _repartir_en_dias(5, {d1: 6, d2: 1}) == {d1: 4, d2: 1}             # el día pequeño se llena; el resto va al otro
assert _repartir_en_dias(2, {d1: 6, d2: 6, d3: 6}) == {d1: 1, d2: 1, d3: 0}
assert _repartir_en_dias(0, {}) == {}


# _a_sesiones
def ses(bloques, mx):
    return [(s.inicio.strftime("%H:%M"), s.fin.strftime("%H:%M")) for s in _a_sesiones("x", d1, bloques, mx)]


assert ses([H(18), H(18, 30), H(19)], 2) == [("18:00", "19:00"), ("19:00", "19:30")]
assert ses([H(18), H(18, 30), H(19), H(19, 30)], 8) == [("18:00", "20:00")]
assert ses([H(16), H(16, 30), H(19)], 8) == [("16:00", "17:00"), ("19:00", "19:30")]    # no cruza el hueco entre franjas
assert ses([H(18)], 1) == [("18:00", "18:30")]
assert ses([], 2) == []

# asignar: una predicción por actividad
plan_min = PlanEntrada(actividades=[dict(nombre="T", prioridad="alta", dificultad="alta", fecha_limite="2026-11-05", horas_estimadas=2)],
                       disponibilidad=[{"dia": "2026-11-01", "franjas": franjas(("18:00", "21:00"))}], duracion_sesion=60)
try:
    asignar(plan_min, [], date(2026, 11, 1))
    raise AssertionError("debía rechazar la falta de predicciones")
except ValueError as e:
    assert "una predicción por actividad" in str(e)

print("Funciones auxiliares: todas las pruebas pasaron")

Funciones auxiliares: todas las pruebas pasaron


# PASO 4. UN EJEMPLO COMPLETO

Dos actividades con 3 h de disponibilidad por día (18:00-21:00) y sesión máxima de 60 min. Primero pasan por el validador y luego por la asignación. Las predicciones se escriben a mano, como si las hubiera entregado el MLP.

* **Parcial de IA:** 8 h, 6 días, prioridad alta y dificultad alta -> `prioridad 0.87`, perfil `[0.45, 0.35, 0.20]` (adelantar el trabajo).
* **Taller:** 4 h, 5 días -> `prioridad 0.55`, perfil `[0.30, 0.40, 0.30]`.

In [8]:
from datetime import timedelta

import pandas as pd

INICIO = date(2026, 10, 25)  # domingo


def f(n):
    return (INICIO + timedelta(days=n)).isoformat()


def act(nombre, n_limite, horas, prioridad="media", dificultad="media"):
    return {"nombre": nombre, "prioridad": prioridad, "dificultad": dificultad, "fecha_limite": f(n_limite), "horas_estimadas": horas}


def dias(desde, hasta, pares=(("18:00", "21:00"),)):
    return [{"dia": f(n), "franjas": franjas(*pares)} for n in range(desde, hasta)]


def plan(actividades, disponibilidad, sesion=60, max_horas_dia=None):
    return {"actividades": actividades, "disponibilidad": disponibilidad, "duracion_sesion": sesion, "max_horas_dia": max_horas_dia}


def pr(prioridad, perfil):
    return Prediccion(prioridad=prioridad, perfil=perfil)


def tabla_sesiones(r):
    return pd.DataFrame([{"actividad": s.actividad, "día": f"{s.dia:%a %d} (n={(s.dia - INICIO).days})",
                          "inicio": f"{s.inicio:%H:%M}", "fin": f"{s.fin:%H:%M}"} for s in r.sesiones])


ejemplo = PlanEntrada(**plan([act("Parcial de IA", 6, 8, "alta", "alta"), act("Taller", 5, 4)], dias(0, 10)))
print("Validador:", validar(ejemplo, INICIO).estado.value)

r = asignar(ejemplo, [pr(0.87, [0.45, 0.35, 0.20]), pr(0.55, [0.30, 0.40, 0.30])], INICIO)
print(pd.DataFrame([a.model_dump(mode="json") for a in r.actividades]).to_string(index=False))
tabla_sesiones(r)

Validador: factible
       nombre  prioridad  horas_solicitadas  horas_asignadas  horas_faltantes   estado
Parcial de IA       0.87                8.0              8.0              0.0 asignada
       Taller       0.55                4.0              4.0              0.0 asignada


,actividad,día,inicio,fin
0,Parcial de IA,Sun 25 (n=0),18:00,19:00
1,Parcial de IA,Sun 25 (n=0),19:00,20:00
2,Taller,Sun 25 (n=0),20:00,21:00
3,Parcial de IA,Mon 26 (n=1),18:00,19:00
4,Parcial de IA,Mon 26 (n=1),19:00,19:30
5,Taller,Mon 26 (n=1),19:30,20:00
6,Parcial de IA,Tue 27 (n=2),18:00,19:00
7,Parcial de IA,Tue 27 (n=2),19:00,19:30
8,Taller,Tue 27 (n=2),19:30,20:30
9,Parcial de IA,Wed 28 (n=3),18:00,19:00


**Observaciones**

1. El validador deja pasar el plan (`factible`) y la asignación ubica las 12 h pedidas: las dos actividades quedan `asignada`.
2. **Parcial de IA** (prioridad 0.87) escoge primero, por eso ocupa el inicio de cada día (18:00). Sus horas por tramo son las del apartado 1.2: **3 h 30** en n=0 y n=1 (2 h + 1 h 30), **3 h** en n=2 y n=3 (1 h 30 cada día) y **1 h 30** en n=4 y n=5 (1 h + 30 min).
3. **Taller** (prioridad 0.55) se acomoda en lo que queda de cada día y reparte 4 h como 1 h 30, 1 h 30 y 1 h. Como vence antes (n=5), no trabaja el viernes.
4. Ninguna sesión pasa de 60 min (`duracion_sesion`) y las que sobran de un día quedan de 30 min.
5. `ResultadoAsignacion` trae exactamente lo que el front necesita para pintar el calendario y lo que el LLM redactor y el validador de salida necesitan para hablar del horario sin inventar nada.

# PASO 5. CASOS DE PRUEBA

Cada caso guarda **entrada, salida esperada y salida lograda**, y esta tabla alimenta la sección 7 del informe. Los casos son datos (`CASOS`), así que después se reutilizan con `pytest.mark.parametrize` sin reescribirlos.

Todos usan `INICIO` = domingo 25 de octubre de 2026 y, salvo que se diga otra cosa, 3 h de disponibilidad por día (18:00-21:00) y sesión de 60 min. `n` es el día respecto al inicio (0 = domingo). La entrada incluye las predicciones del MLP, escritas a mano.

## 5.1 Reglas que todo resultado debe cumplir

`verificar` se ejecuta en **todos** los casos y en las pruebas aleatorias. Comprueba el objetivo del proyecto de **0 sesiones fuera de la disponibilidad**, además de las fechas, la duración de las sesiones, el máximo diario y la coherencia del resumen.

In [9]:
def verificar(plan_dict, r, fecha_inicio, ocupadas=()):
    p = PlanEntrada(**plan_dict)
    por_nombre = {a.nombre: a for a in p.actividades}
    franjas_de = {d.dia: d.franjas for d in p.disponibilidad}
    horas = dict.fromkeys(por_nombre, 0.0)
    for s in r.sesiones:
        a = por_nombre[s.actividad]
        ini, fin = _minutos(s.inicio), _minutos(s.fin)
        assert fecha_inicio <= s.dia < a.fecha_limite, f"fuera de fechas: {s}"
        assert BLOQUE_MIN <= fin - ini <= p.duracion_sesion and (fin - ini) % BLOQUE_MIN == 0, f"duración inválida: {s}"
        for b in range(ini, fin, BLOQUE_MIN):   # cada bloque debe estar dentro de una franja
            assert any(_minutos(fr.inicio) <= b and b + BLOQUE_MIN <= _minutos(fr.fin) for fr in franjas_de.get(s.dia, [])), f"fuera de disponibilidad: {s}"
        horas[s.actividad] += (fin - ini) / 60
    todas = sorted(list(r.sesiones) + list(ocupadas), key=lambda s: (s.dia, s.inicio))
    for x, y in zip(todas, todas[1:]):
        assert x.dia != y.dia or x.fin <= y.inicio, f"sesiones traslapadas: {x} / {y}"
    if p.max_horas_dia is not None:
        for d in {s.dia for s in todas}:
            total = sum((_minutos(s.fin) - _minutos(s.inicio)) / 60 for s in todas if s.dia == d)
            assert total <= p.max_horas_dia + 1e-9, f"supera el máximo diario el {d}"
    assert len(r.actividades) == len(p.actividades)
    for a in r.actividades:
        assert abs(horas[a.nombre] - a.horas_asignadas) < 1e-9, f"el resumen no coincide con las sesiones: {a.nombre}"
        assert (a.estado == EstadoAsignacion.ASIGNADA) == (a.horas_faltantes == 0)
        assert (a.estado == EstadoAsignacion.NO_ASIGNADA) == (a.horas_asignadas == 0)
    assert all(x.prioridad >= y.prioridad for x, y in zip(r.actividades, r.actividades[1:]))   # resumen en orden de prioridad

## 5.2 Casos

In [10]:
def hh(h):
    return f"{h:g}"


def esp(estados, horas_dia=None, sesiones=None):
    return {"estados": estados, "horas_dia": horas_dia, "sesiones": sesiones}


UNIF = [1/3] * 3
OC = Sesion(actividad="Fijada", dia=f(0), inicio="18:00", fin="19:00")

CASOS = [
    {"nombre": "Reparto del ejemplo (8 h)", "resumen": "8 h, 6 días, perfil [0.45, 0.35, 0.20]: 3 h 30 / 3 h / 1 h 30",
     "entrada": plan([act("Parcial", 6, 8, "alta", "alta")], dias(0, 10)), "pred": [pr(0.87, [0.45, 0.35, 0.20])],
     "esperado": esp({"Parcial": "asignada"}, {0: 2, 1: 1.5, 2: 1.5, 3: 1.5, 4: 1, 5: 0.5})},
    {"nombre": "Manda la prioridad, no el orden de la lista", "resumen": "dos de 3 h compiten por 18:00-20:00; 'Taller B' va primero en la lista pero con prioridad 0.9",
     "entrada": plan([act("Taller B", 3, 3), act("Informe A", 3, 3)], dias(0, 3, (("18:00", "20:00"),))), "pred": [pr(0.9, UNIF), pr(0.4, UNIF)],
     "esperado": esp({"Taller B": "asignada", "Informe A": "asignada"}, {0: 2, 1: 2, 2: 2},
                     [(t, n, i, e) for n in range(3) for t, i, e in (("Taller B", "18:00", "19:00"), ("Informe A", "19:00", "20:00"))])},
    {"nombre": "Primer tramo sin espacio", "resumen": "días n=0 y n=1 de solo 30 min; sobra del primer tramo pasa al segundo",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 2, (("18:00", "18:30"),)) + dias(2, 10)), "pred": [pr(0.7, [0.5, 0.3, 0.2])],
     "esperado": esp({"Proyecto": "asignada"}, {0: 0.5, 1: 0.5, 2: 1, 3: 1, 4: 0.5, 5: 0.5})},
    {"nombre": "Último tramo sin espacio", "resumen": "días n=4 y n=5 de solo 30 min; la sobra del último tramo pasa al segundo",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 4) + dias(4, 6, (("18:00", "18:30"),)) + dias(6, 10)), "pred": [pr(0.7, [0.2, 0.3, 0.5])],
     "esperado": esp({"Proyecto": "asignada"}, {0: 0.5, 1: 0.5, 2: 1, 3: 1, 4: 0.5, 5: 0.5})},
    {"nombre": "Tramo del medio sin espacio", "resumen": "días n=2 y n=3 de solo 30 min; la sobra pasa al tramo posterior, no al anterior",
     "entrada": plan([act("Proyecto", 6, 4)], dias(0, 2) + dias(2, 4, (("18:00", "18:30"),)) + dias(4, 10)), "pred": [pr(0.7, [0.3, 0.4, 0.3])],
     "esperado": esp({"Proyecto": "asignada"}, {0: 1, 1: 0.5, 2: 0.5, 3: 0.5, 4: 1, 5: 0.5})},
    {"nombre": "Solo cabe una parte", "resumen": "6 h de trabajo y 3 h libres (1 h por día): se asigna lo que cabe",
     "entrada": plan([act("Tesis", 3, 6)], dias(0, 3, (("18:00", "19:00"),))), "pred": [pr(0.8, UNIF)],
     "esperado": esp({"Tesis": "parcial (faltan 3 h)"}, {0: 1, 1: 1, 2: 1})},
    {"nombre": "Sin espacio para la de menor prioridad", "resumen": "3 h libres; 'Examen' (prioridad 0.9) las toma todas y 'Lectura' queda fuera",
     "entrada": plan([act("Examen", 3, 3), act("Lectura", 3, 2)], dias(0, 3, (("18:00", "19:00"),))), "pred": [pr(0.9, UNIF), pr(0.3, UNIF)],
     "esperado": esp({"Examen": "asignada", "Lectura": "no_asignada (faltan 2 h)"}, {0: 1, 1: 1, 2: 1})},
    {"nombre": "Máximo diario de 1 h", "resumen": "4 h en 5 días de 3 h libres, pero máximo de 1 h por día",
     "entrada": plan([act("Proyecto", 5, 4)], dias(0, 10), max_horas_dia=1), "pred": [pr(0.6, UNIF)],
     "esperado": esp({"Proyecto": "asignada"}, {0: 1, 1: 0.5, 2: 1, 3: 0.5, 4: 1})},
    {"nombre": "Sesiones de máximo 90 min", "resumen": "4 h el mismo día, franja 18:00-22:00: 90 + 90 + 60 min",
     "entrada": plan([act("Informe", 3, 4)], dias(0, 10, (("18:00", "22:00"),)), sesion=90), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 4}, [("Informe", 0, "18:00", "19:30"), ("Informe", 0, "19:30", "21:00"), ("Informe", 0, "21:00", "22:00")])},
    {"nombre": "Dos franjas en el mismo día", "resumen": "franjas 16:00-17:30 y 19:00-20:30; la sesión no cruza el hueco",
     "entrada": plan([act("Informe", 3, 2)], dias(0, 10, (("16:00", "17:30"), ("19:00", "20:30"))), sesion=90), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 2}, [("Informe", 0, "16:00", "17:30"), ("Informe", 0, "19:00", "19:30")])},
    {"nombre": "Franja que no es múltiplo de 30 min", "resumen": "franja 18:00-19:50 aporta 1 h 30; la media hora restante pasa al día siguiente",
     "entrada": plan([act("Informe", 3, 2)], dias(0, 10, (("18:00", "19:50"),)), sesion=120), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Informe": "asignada"}, {0: 1.5, 1: 0.5}, [("Informe", 0, "18:00", "19:30"), ("Informe", 1, "18:00", "18:30")])},
    {"nombre": "Replanificación: lo ocupado se respeta", "resumen": "hay una sesión fija el día n=0 de 18:00 a 19:00", "ocupadas": [OC],
     "entrada": plan([act("Taller", 3, 2)], dias(0, 10)), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 2}, [("Taller", 0, "19:00", "20:00"), ("Taller", 0, "20:00", "21:00")])},
    {"nombre": "Lo ocupado cuenta para el máximo diario", "resumen": "igual al anterior con máximo de 2 h: queda 1 h el día n=0 y el resto pasa al siguiente", "ocupadas": [OC],
     "entrada": plan([act("Taller", 3, 2)], dias(0, 10), max_horas_dia=2), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 1, 1: 1}, [("Taller", 0, "19:00", "20:00"), ("Taller", 1, "18:00", "19:00")])},
    {"nombre": "Ventana de 2 días (replanificación)", "resumen": "3 h con solo n=0 y n=1; el tercer tramo no tiene días y entrega al segundo",
     "entrada": plan([act("Quiz", 2, 3)], dias(0, 10)), "pred": [pr(0.6, [0.4, 0.3, 0.3])],
     "esperado": esp({"Quiz": "asignada"}, {0: 1, 1: 2})},
    {"nombre": "Fecha límite vencida", "resumen": "'Vieja' vence en n=0 (sin días de trabajo); 'Taller' se asigna normal",
     "entrada": plan([act("Vieja", 0, 2), act("Taller", 5, 2)], dias(0, 10)), "pred": [pr(0.9, UNIF), pr(0.5, UNIF)],
     "esperado": esp({"Vieja": "no_asignada (faltan 2 h)", "Taller": "asignada"}, {0: 0.5, 1: 0.5, 2: 0.5, 4: 0.5})},
    {"nombre": "Disponibilidad anterior al inicio", "resumen": "hay disponibilidad desde n=-2; se ignora",
     "entrada": plan([act("Taller", 6, 2)], dias(-2, 10)), "pred": [pr(0.5, UNIF)],
     "esperado": esp({"Taller": "asignada"}, {0: 0.5, 1: 0.5, 2: 0.5, 4: 0.5})},
    {"nombre": "Horas que no son múltiplo de 30 min", "resumen": "2.2 h se redondean a 2 h (4 bloques)",
     "entrada": plan([act("Taller", 3, 2.2)], dias(0, 10)), "pred": [pr(0.5, [1, 0, 0])],
     "esperado": esp({"Taller": "asignada"}, {0: 2})},
]


def ejecutar(caso):
    p = PlanEntrada(**caso["entrada"])
    ocupadas = caso.get("ocupadas", [])
    r = asignar(p, caso["pred"], INICIO, ocupadas)
    verificar(caso["entrada"], r, INICIO, ocupadas)
    estados = {a.nombre: a.estado.value + ("" if a.estado.value == "asignada" else f" (faltan {hh(a.horas_faltantes)} h)") for a in r.actividades}
    horas_dia = {}
    for s in r.sesiones:
        n = (s.dia - INICIO).days
        horas_dia[n] = horas_dia.get(n, 0) + (_minutos(s.fin) - _minutos(s.inicio)) / 60
    sesiones = [(s.actividad, (s.dia - INICIO).days, f"{s.inicio:%H:%M}", f"{s.fin:%H:%M}") for s in r.sesiones]
    return esp(estados, dict(sorted(horas_dia.items())), sesiones)


def coincide(esperado, lograda):
    return (esperado["estados"] == lograda["estados"]
            and esperado["horas_dia"] == lograda["horas_dia"]
            and (esperado["sesiones"] is None or esperado["sesiones"] == lograda["sesiones"]))


def texto(s):
    estados = ", ".join(f"{n}: {e}" for n, e in s["estados"].items())
    horas = " ".join(f"n{n}={hh(h)}" for n, h in s["horas_dia"].items())
    return f"{estados} | horas/día: {horas}"


filas = []
for c in CASOS:
    lograda = ejecutar(c)
    filas.append({"Caso": c["nombre"], "Entrada": c["resumen"], "Esperada": texto(c["esperado"]),
                  "Lograda": texto(lograda), "OK": "sí" if coincide(c["esperado"], lograda) else "NO"})

pd.set_option("display.max_colwidth", 120)
tabla = pd.DataFrame(filas)
print(f"{(tabla['OK'] == 'sí').sum()} de {len(tabla)} casos coinciden con lo esperado")
tabla

17 de 17 casos coinciden con lo esperado


,Caso,Entrada,Esperada,Lograda,OK
0,Reparto del ejemplo (8 h),"8 h, 6 días, perfil [0.45, 0.35, 0.20]: 3 h 30 / 3 h / 1 h 30",Parcial: asignada | horas/día: n0=2 n1=1.5 n2=1.5 n3=1.5 n4=1 n5=0.5,Parcial: asignada | horas/día: n0=2 n1=1.5 n2=1.5 n3=1.5 n4=1 n5=0.5,sí
1,"Manda la prioridad, no el orden de la lista",dos de 3 h compiten por 18:00-20:00; 'Taller B' va primero en la lista pero con prioridad 0.9,"Taller B: asignada, Informe A: asignada | horas/día: n0=2 n1=2 n2=2","Taller B: asignada, Informe A: asignada | horas/día: n0=2 n1=2 n2=2",sí
2,Primer tramo sin espacio,días n=0 y n=1 de solo 30 min; sobra del primer tramo pasa al segundo,Proyecto: asignada | horas/día: n0=0.5 n1=0.5 n2=1 n3=1 n4=0.5 n5=0.5,Proyecto: asignada | horas/día: n0=0.5 n1=0.5 n2=1 n3=1 n4=0.5 n5=0.5,sí
3,Último tramo sin espacio,días n=4 y n=5 de solo 30 min; la sobra del último tramo pasa al segundo,Proyecto: asignada | horas/día: n0=0.5 n1=0.5 n2=1 n3=1 n4=0.5 n5=0.5,Proyecto: asignada | horas/día: n0=0.5 n1=0.5 n2=1 n3=1 n4=0.5 n5=0.5,sí
4,Tramo del medio sin espacio,"días n=2 y n=3 de solo 30 min; la sobra pasa al tramo posterior, no al anterior",Proyecto: asignada | horas/día: n0=1 n1=0.5 n2=0.5 n3=0.5 n4=1 n5=0.5,Proyecto: asignada | horas/día: n0=1 n1=0.5 n2=0.5 n3=0.5 n4=1 n5=0.5,sí
5,Solo cabe una parte,6 h de trabajo y 3 h libres (1 h por día): se asigna lo que cabe,Tesis: parcial (faltan 3 h) | horas/día: n0=1 n1=1 n2=1,Tesis: parcial (faltan 3 h) | horas/día: n0=1 n1=1 n2=1,sí
6,Sin espacio para la de menor prioridad,3 h libres; 'Examen' (prioridad 0.9) las toma todas y 'Lectura' queda fuera,"Examen: asignada, Lectura: no_asignada (faltan 2 h) | horas/día: n0=1 n1=1 n2=1","Examen: asignada, Lectura: no_asignada (faltan 2 h) | horas/día: n0=1 n1=1 n2=1",sí
7,Máximo diario de 1 h,"4 h en 5 días de 3 h libres, pero máximo de 1 h por día",Proyecto: asignada | horas/día: n0=1 n1=0.5 n2=1 n3=0.5 n4=1,Proyecto: asignada | horas/día: n0=1 n1=0.5 n2=1 n3=0.5 n4=1,sí
8,Sesiones de máximo 90 min,"4 h el mismo día, franja 18:00-22:00: 90 + 90 + 60 min",Informe: asignada | horas/día: n0=4,Informe: asignada | horas/día: n0=4,sí
9,Dos franjas en el mismo día,franjas 16:00-17:30 y 19:00-20:30; la sesión no cruza el hueco,Informe: asignada | horas/día: n0=2,Informe: asignada | horas/día: n0=2,sí


In [11]:
assert (tabla["OK"] == "sí").all(), tabla[tabla["OK"] == "NO"]

**Observaciones**

1. Los tres casos de tramo sin espacio cubren las tres posiciones posibles: el excedente del primer tramo pasa al segundo, el del último pasa al anterior y el del medio pasa al **posterior** (la regla de desempate).
2. En los dos casos de espacio insuficiente la actividad no se fuerza: queda `parcial` con las horas faltantes, o `no_asignada`. La prioridad decide quién se queda con el espacio (`Examen` sobre `Lectura`).
3. En "Manda la prioridad" la lista trae primero a la actividad más prioritaria, pero el caso se diseñó para que el orden de la lista no importe: lo que cuenta es la prioridad del MLP.
4. Los casos de replanificación comprueban que lo ya ocupado se respeta y que cuenta para el máximo diario, y que una ventana de menos de 3 días sigue funcionando (el tramo sin días entrega a su vecino).
5. `verificar` corrió dentro de cada caso: en los 17 hay **0 sesiones fuera de la disponibilidad**, ninguna en o después de la fecha límite ni antes del inicio, ninguna traslapada y ningún día sobre el máximo.

## 5.3 Prueba con planes aleatorios

Genera planes al azar (1 a 5 actividades, franjas y topes variados), se queda con los que el validador declara **factibles** y les asigna prioridades y perfiles aleatorios, es decir, sin la ayuda del MLP. En todos se ejecuta `verificar`, y se cuenta cuántos quedan **completamente asignados**.

In [12]:
rng = random.Random(7)
NIV = ["baja", "media", "alta"]
FRANJAS = [(("18:00", "21:00"),), (("08:00", "10:00"), ("14:00", "16:30")), (("19:00", "20:30"),), (("16:00", "17:50"), ("19:00", "21:00"))]


def plan_aleatorio():
    actividades = [act(f"A{i}", rng.randint(3, 14), rng.choice([1, 1.5, 2, 3, 4, 5, 6]), rng.choice(NIV), rng.choice(NIV))
                   for i in range(rng.randint(1, 5))]
    disponibilidad = [{"dia": f(n), "franjas": franjas(*rng.choice(FRANJAS))} for n in range(16) if rng.random() < 0.85]
    return plan(actividades, disponibilidad, rng.choice([60, 90, 120]), rng.choice([None, 3, 4]))


def prediccion_aleatoria():
    w = [rng.random() for _ in range(3)]
    return pr(rng.random(), [x / sum(w) for x in w])


factibles = completos = 0
for _ in range(2000):
    d = plan_aleatorio()
    if not d["disponibilidad"]:
        continue
    p = PlanEntrada(**d)
    if validar(p, INICIO).estado == Estado.INVIABLE:
        continue
    factibles += 1
    r = asignar(p, [prediccion_aleatoria() for _ in p.actividades], INICIO)
    verificar(d, r, INICIO)
    completos += all(a.estado == EstadoAsignacion.ASIGNADA for a in r.actividades)

print(f"{factibles} planes factibles; todas las reglas de verificar se cumplieron en los {factibles}")
print(f"{completos} quedaron completamente asignados ({completos / factibles:.1%})")

1732 planes factibles; todas las reglas de verificar se cumplieron en los 1732
1655 quedaron completamente asignados (95.6%)


**Observaciones**

1. De los 2000 planes generados (semilla 7), 1732 los acepta el validador. En los 1732 se cumplieron todas las reglas de `verificar`: **0 sesiones fuera de la disponibilidad** también con planes que ningún caso escrito a mano anticipó.
2. 1655 (95.6 %) quedaron completamente asignados. En el 4.4 % restante alguna actividad quedó `parcial` o `no_asignada` aunque el validador declaró el plan factible: es el límite conocido del paso 6 (la prioridad aleatoria deja que una actividad de plazo lejano ocupe bloques que otra de plazo cercano necesitaba). En todos esos casos la capa lo reporta en vez de forzarlo.

# PASO 6. MIGRACIÓN A ARCHIVOS `.py`

| Celdas del notebook | Archivo | Imports que se agregan |
|---|---|---|
| Paso 2, celda de modelos, y Paso 3, celda del algoritmo | `studyflow/asignacion.py` (una sola, en ese orden) | `from models import Actividad, DisponibilidadDia, PlanEntrada` y `from disponibilidad import BLOQUE_MIN, minutos` |
| Pasos 2.1, 3.1, 4 y 5 | `tests/` | los helpers (`act`, `dias`, `plan`, `dia`, `pr`) y `verificar` van a `conftest.py`; `CASOS` se reutiliza con `pytest.mark.parametrize` |

Cuando la lógica esté en `.py`, el notebook importa desde los módulos y deja de definirla.

Al migrar, en `disponibilidad.py` conviene renombrar `_minutos` a `minutos`, porque ahora lo usan otros módulos. No cambia nada más.

## Puntos para tener en cuenta en las otras capas

* **Conexión con el MLP:** por cada actividad, en el mismo orden que `plan.actividades`, se crea `Prediccion(prioridad=float(salida_a[i]), perfil=[float(x) for x in salida_b[i]])`. La conversión a `float` es necesaria porque Keras entrega `float32`.
* **Front y LLM redactor:** `ResultadoAsignacion.model_dump_json()` entrega todo lo necesario. El validador de salida compara el texto del LLM con `sesiones` y `actividades` (en especial las `parcial` y `no_asignada`, que el LLM debe mencionar y no ocultar).
* **Replanificación:** quien llame a `asignar` arma el `PlanEntrada` solo con lo pendiente (`horas_estimadas` = horas que faltan), pone `fecha_inicio` en el primer día replanificable y pasa en `ocupadas` las sesiones ya cumplidas o fijadas de esos días. El validador corre antes en modo `"replanificacion"`.
* **Límite conocido:** la asignación por prioridad no garantiza ubicar todo lo que el validador declaró factible. Si una actividad de prioridad alta y plazo lejano toma bloques que una de plazo cercano necesitaba, la segunda queda `parcial` o `no_asignada` aunque el plan completo cupiera. Siempre se reporta, nunca se oculta. La prueba 5.3 mide qué tan frecuente es con prioridades aleatorias (independientes del plazo); queda por medirla con las prioridades reales del MLP, que sí incluyen la urgencia.